[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_primitives_gc.ipynb)

# Sólidos Primitivos e Operações Booleanas
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

---

Neste notebook vamos explorar as **formas primitivas** do build123d e as **operações de modelagem** que permitem criar geometrias arquitetônicas mais complexas.

---

## Instalação

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d]"],
        check=True,
    )
    # build123d pulls in a newer ipython than Colab's kernel bootstrap
    # tolerates. Put Colab's version back on disk — do NOT restart the
    # runtime, the current kernel already has the working ipython loaded.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )

else:
    print("Not running in Colab, skipping package installation.")


## Importações

In [ ]:
import build123d as b3d
from cadquery_simple_viewer import show

---

## Sólidos Primitivos

Os **sólidos primitivos** são formas geométricas básicas que o build123d consegue criar diretamente. São o ponto de partida para construções mais complexas.

---

### Caixa — `Box()`

A caixa retangular é a primitiva mais usada em arquitetura. Representa qualquer volume ortogonal: paredes, lajes, pilares retangulares, blocos.

```python
Box(length, width, height)
```

| Parâmetro | Eixo | Descrição |
|-----------|------|-----------|
| `length` | X | Comprimento |
| `width`  | Y | Largura |
| `height` | Z | Altura |

Por padrão, a caixa é criada **centrada na origem**. O parâmetro `align` controla esse comportamento — ele recebe uma tupla de três valores `Align.MIN`, `Align.CENTER` ou `Align.MAX`, um para cada eixo:
- `align=(Align.CENTER, Align.CENTER, Align.CENTER)` → centro geométrico na origem (padrão)
- `align=(Align.MIN, Align.MIN, Align.MIN)` → canto inferior esquerdo na origem — equivalente ao `centered=False` do CadQuery

In [ ]:
pilar_centrado = b3d.Box(0.4, 0.4, 3.0)

pilar_canto = b3d.Box(0.4, 0.4, 3.0, align=(b3d.Align.MIN, b3d.Align.MIN, b3d.Align.MIN))
pilar_canto = pilar_canto.translate((2.0, 0, 0))

show(
    [pilar_centrado, pilar_canto],
    names=["Centrado", "Canto na origem"],
    visible_axes=None,
    z=0,
    plane_size=6
)

---

### Cilindro — `Cylinder()`

O cilindro representa pilares circulares, colunas, tanques, torres cilíndricas.

```python
Cylinder(radius, height, arc_size=360)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `radius` | Raio da base |
| `height` | Altura |
| `arc_size`  | Ângulo de abertura em graus — padrão `360` (cilindro completo) |

> ⚠️ **Atenção à ordem**: no CadQuery, `.cylinder(height, radius)` recebe a altura primeiro. No build123d, `b3d.Cylinder(radius, height)` recebe o **raio primeiro** — use argumentos nomeados para evitar trocar os dois.

O parâmetro `arc_size` permite criar **segmentos de cilindro**, úteis para curvas parciais e elementos curvos.

In [ ]:
coluna = b3d.Cylinder(radius=0.2, height=3.0)

parede_curva = b3d.Cylinder(radius=1.5, height=2.8, arc_size=90)
parede_curva = parede_curva.translate((4.0, 0, 0))

parede_semi = b3d.Cylinder(radius=2.5, height=2.8, arc_size=180)
parede_semi = parede_semi.translate((10.0, 0, 0))

show(
    [coluna, parede_curva, parede_semi],
    names=["Coluna", "Parede curva 90°", "Parede semicircular"],
    visible_axes=None,
    z=0,
    plane_size=20
)

---

### Esfera — `Sphere()`

A esfera é usada em cúpulas, elementos decorativos, volumes orgânicos e estudo de formas.

```python
Sphere(radius, arc_size1=-90, arc_size2=90, arc_size3=360)
```

| Parâmetro  | Descrição |
|-----------|-----------|
| `radius`  | Raio |
| `arc_size1`  | Ângulo inicial na vertical (latitude inferior) — padrão `-90°` |
| `arc_size2`  | Ângulo final na vertical (latitude superior) — padrão `90°` |
| `arc_size3`  | Ângulo de abertura horizontal — padrão `360°` |

Controlando `arc_size1` e `arc_size2` é possível criar **calotas esféricas** — como cúpulas.

In [ ]:
esfera = b3d.Sphere(2.0)

cupula = b3d.Sphere(2.0, arc_size1=0, arc_size2=90)
cupula = cupula.translate((6.0, 0, 0))

cupula_rasa = b3d.Sphere(2.0, arc_size1=60, arc_size2=90)
cupula_rasa = cupula_rasa.translate((12.0, 0, 0))

show(
    [esfera, cupula, cupula_rasa],
    names=["Esfera completa", "Cúpula", "Cúpula rasa"],
    visible_axes=None,
    z=-2.0,
    plane_size=18
)

---

### Cone e Tronco de Cone — `Cone()`

Diferente do CadQuery — que precisa recorrer diretamente ao OCCT via `BRepPrimAPI_MakeCone` para criar cones — o build123d já oferece o cone como **primitiva de primeira classe**:

```python
Cone(bottom_radius, top_radius, height)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `bottom_radius` | Raio da base |
| `top_radius`    | Raio do topo — `0` gera um cone pontiagudo; `> 0` gera um tronco de cone |
| `height`        | Altura |

In [ ]:
# --- Cone pontiagudo ---
telhado_conico = b3d.Cone(bottom_radius=2.5, top_radius=0, height=2.0)

# --- Tronco de cone — base maior em baixo ---
tronco = b3d.Cone(bottom_radius=3.0, top_radius=1.5, height=3.0)
tronco = tronco.translate((8.0, 0, 0))

# --- Tronco invertido — base menor em baixo ---
tronco_invertido = b3d.Cone(bottom_radius=1.0, top_radius=3.0, height=2.0)
tronco_invertido = tronco_invertido.translate((16.0, 0, 0))

show(
    [telhado_conico, tronco, tronco_invertido],
    names=["Telhado cônico", "Tronco de cone", "Tronco invertido"],
    visible_axes=None,
    z=-1.5,
    plane_size=25
)

---

### Superfície de Revolução — `revolve()` e Toro

O **revolve** (revolução) cria um sólido girando um perfil 2D em torno de um eixo. É a operação adequada para qualquer forma com **simetria axial**: coberturas anulares, colunas com entases, formas de vaso.

O processo é sempre o mesmo:
1. Desenhar o perfil em um `BuildLine`, dentro de um `BuildSketch` no plano que contém o perfil e o eixo de revolução
2. Fechar o perfil em uma face com `make_face()`
3. Chamar `revolve(perfil, eixo)` — por padrão o eixo é `b3d.Axis.Z`

**Limitação do OCCT**: revolucionar um perfil **circular** 360° em torno de um eixo coplanar (o caso do toro) causa uma falha numérica interna do motor geométrico — essa limitação vem do próprio OCCT, e por isso também existe no build123d. Para criar um toro, usamos a primitiva dedicada **`Torus()`**, que não passa pelo caminho do `revolve()` e não sofre desse problema:

```python
Torus(major_radius, minor_radius)
```

| Parâmetro | Descrição |
|-----------|-----------|
| `major_radius` | raio do anel central (distância do centro ao tubo) |
| `minor_radius` | raio do tubo (espessura do anel) |

In [ ]:
# --- Toro (argola) via Torus() ---
r1 = 3.0
r2 = 0.5

anel = b3d.Torus(major_radius=r1, minor_radius=r2)

# --- Anel com tubo mais espesso ---
r1_grosso = 3.0
r2_grosso = 1.2

anel_grosso = b3d.Torus(major_radius=r1_grosso, minor_radius=r2_grosso)
anel_grosso = anel_grosso.translate((10.0, 0, 0))

show(
    [anel, anel_grosso],
    names=["Anel fino", "Anel grosso"],
    visible_axes=None,
    z=-0.6,
    plane_size=18
)

O revolve não precisa ser circular. Qualquer perfil 2D fechado pode ser girado em torno de um eixo:

In [ ]:
# Perfil trapezoidal girado 360° → forma cônica com espessura

dist_eixo = 0.5   # distância mínima do perfil ao eixo
larg_base = 2.0   # largura da base do perfil
larg_topo = 1.0   # largura do topo do perfil
altura_p  = 3.0   # altura do perfil

with b3d.BuildSketch(b3d.Plane.XZ) as perfil_sketch:
    with b3d.BuildLine() as perfil_linha:
        b3d.Polyline(
            (dist_eixo,             0),
            (dist_eixo + larg_base, 0),
            (dist_eixo + larg_topo, altura_p),
            (dist_eixo,             altura_p),
            close=True,
        )
    b3d.make_face()

forma_revolvida = b3d.revolve(perfil_sketch.sketch, b3d.Axis.Z)

show(
    [forma_revolvida],
    names=["Forma revolvida"],
    visible_axes=None,
    z=0,
    plane_size=8
)

---

### Rampa — via Loft entre Perfis Paralelos

O loft entre **dois perfis idênticos em posições diferentes** cria uma transição inclinada — ideal para rampas. O perfil do topo é uma cópia da base deslocada para a outra extremidade e elevada.

> ⚠️ **Nota de conversão importante**: no CadQuery, `.loft()` aceita wires "pendentes" diretamente — o motor fecha e triangula a superfície internamente. No build123d, `loft()` exige **faces** (`Face`/`Sketch`) como seções — um `Wire` sozinho não tem `.faces()` e é silenciosamente ignorado (o loft falha reclamando de menos de duas seções). Por isso cada perfil da rampa aqui é um pequeno retângulo (`Rectangle`) — a **seção transversal** da rampa (largura x espessura) — posicionado com um `b3d.Plane` cujo eixo Z aponta na direção do percurso (X). O loft entre as duas seções produz um **volume sólido** de rampa, e não apenas uma superfície.

In [ ]:
# --- Rampa via loft ---

dx_r = 5.0   # comprimento da rampa
dy_r = 3.0   # largura da rampa
dz_r = 1.5   # altura total da rampa
esp_r = 0.2  # espessura da laje da rampa

# Plano com eixo Z apontando no sentido do percurso (X), para orientar a seção
plano_base = b3d.Plane(origin=(-dx_r/2, 0, 0),  x_dir=(0, 1, 0), z_dir=(1, 0, 0))
plano_topo = b3d.Plane(origin=( dx_r/2, 0, dz_r), x_dir=(0, 1, 0), z_dir=(1, 0, 0))

# Perfil da base: seção transversal (largura x espessura) na extremidade inicial
base_rampa = plano_base * b3d.Rectangle(dy_r, esp_r)

# Perfil do topo: mesma seção na outra extremidade, já elevada
topo_rampa = plano_topo * b3d.Rectangle(dy_r, esp_r)

rampa = b3d.loft([base_rampa, topo_rampa])

show(
    [rampa],
    names=["Rampa"],
    visible_axes=None,
    z=0,
    plane_size=8
)

---

## Operações Booleanas

As **operações booleanas** permitem combinar dois ou mais sólidos para criar formas que não seriam possíveis com uma única primitiva.

No build123d elas existem tanto como **métodos** quanto como **operadores** — os operadores são o estilo idiomático do modo Álgebra:

| Operação | Método | Operador | Resultado |
|----------|--------|----------|-----------|
| União | `.fuse(outro)` | `a + b` | Une os dois sólidos em um único volume |
| Subtração | `.cut(outro)` | `a - b` | Remove o volume do segundo sólido do primeiro |
| Interseção | `.intersect(outro)` | `a & b` | Mantém apenas o volume compartilhado pelos dois |

> ⚠️ Note que apenas o nome do método de **união** mudou (`.union()` → `.fuse()`). `.cut()` e `.intersect()` mantêm o mesmo nome do CadQuery.
>
> 💡 **Dica**: nas operações booleanas, a **ordem importa** para subtração e interseção. `a - b` é diferente de `b - a`.

---

### União — `+` / `.fuse()`

In [ ]:
corpo  = b3d.Box(10.0, 8.0, 6.0)
acesso = b3d.Plane(origin=(7.0, 0, -1.5)) * b3d.Box(4.0, 4.0, 3.0)

# Antes da união
show(
    [corpo, acesso],
    names=["Corpo", "Acesso"],
    colors=["lightsteelblue", "lightsalmon"],
    visible_axes=None,
    z=-3.0,
    plane_size=20
)

In [ ]:
edificio = corpo + acesso

show(
    [edificio],
    names=["Edifício unido"],
    visible_axes=None,
    z=-3.0,
    plane_size=20
)

---

### Subtração — `-` / `.cut()`

In [ ]:
parede     = b3d.Box(8.0, 0.3, 3.0)
vao_janela = b3d.Plane(origin=(0, 0, 0.4)) * b3d.Box(1.5, 0.5, 1.5)
vao_porta  = b3d.Plane(origin=(-3.0, 0, -0.5)) * b3d.Box(1.0, 0.5, 2.0)

parede_com_vaos = parede - vao_janela - vao_porta

show(
    [parede_com_vaos],
    names=["Parede com vãos"],
    colors=["lightgray"],
    visible_axes=None,
    z=-1.5,
    plane_size=10
)

In [ ]:
# Pátio escavado em um volume compacto
bloco = b3d.Box(12.0, 12.0, 4.0)
patio = b3d.Plane(origin=(0, 0, 0.5)) * b3d.Box(6.0, 6.0, 5.0)

edificio_patio = bloco - patio

show(
    [edificio_patio],
    names=["Edifício com pátio"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-2.0,
    plane_size=15
)

---

### Interseção — `&` / `.intersect()`

In [ ]:
cilindro_h = b3d.Plane.XZ * b3d.Cylinder(radius=2.0, height=6.0)
cilindro_v = b3d.Cylinder(radius=2.0, height=6.0)

# Antes da interseção — transparência para ver os dois sobrepostos
show(
    [cilindro_h, cilindro_v],
    names=["Cilindro horizontal", "Cilindro vertical"],
    colors=["lightsteelblue", "lightsalmon"],
    opacity=0.6
)

In [ ]:
intersecao = cilindro_h & cilindro_v

show([intersecao], names=["Interseção"], colors=["steelblue"])

In [ ]:
# Envelope máximo x cone de afastamento (via loft)
# loft() exige seções Face/Sketch (não Wire) — por isso usamos Circle()
# posicionado por um Plane, em vez de Wire.make_circle()
envelope = b3d.Box(10.0, 10.0, 8.0)

base_af = b3d.Plane(origin=(0, -7, -4), z_dir=(0, 0, 1)) * b3d.Circle(14.0)
topo_af = b3d.Plane(origin=(0, -7,  8), z_dir=(0, 0, 1)) * b3d.Circle(2.0)

cone_afastamento = b3d.loft([base_af, topo_af])

show(
    [envelope, cone_afastamento],
    names=["Envelope", "Cone de afastamento"],
    colors=["lightsteelblue", "lightsalmon"],
    opacity=0.5,
    visible_axes=None,
    z=-4.0,
    plane_size=20
)

In [ ]:
volume_resultante = envelope & cone_afastamento

show(
    [volume_resultante],
    names=["Volume resultante"],
    colors=["steelblue"],
    visible_axes=None,
    z=-4.0,
    plane_size=15
)

---

## Combinando Operações

Edifício com torre, cobertura cônica e aberturas.

In [ ]:
# --- Dimensões ---
base_larg  = 10.0
base_prof  = 8.0
base_alt   = 4.0
torre_raio = 1.5
torre_alt  = 6.0

# --- Volumes ---
base  = b3d.Box(base_larg, base_prof, base_alt)
torre = b3d.Cylinder(radius=torre_raio, height=torre_alt)

# --- Cobertura cônica ---
z_cob     = torre_alt / 2
cobertura = b3d.Cone(bottom_radius=2.0, top_radius=0.01, height=2.5)
cobertura = cobertura.translate((0, 0, z_cob))

# --- Vãos ---
janela_frente = b3d.Plane(origin=(0, 0, 0.5)) * b3d.Box(2.0, base_prof + 0.5, 1.5)
janela_lado   = b3d.Plane(origin=(0, 0, 0.5)) * b3d.Box(base_larg + 0.5, 2.0, 1.5)

# --- Montagem ---
edificio = base + torre
edificio = edificio + cobertura
edificio = edificio - janela_frente
edificio = edificio - janela_lado

show(
    [edificio],
    names=["Composição final"],
    colors=["lightsteelblue"],
    visible_axes=None,
    z=-2.0,
    plane_size=15
)

---

## Exercício

Crie um modelo volumétrico de um **pavilhão simples** com os seguintes elementos:

1. **Base**: volume retangular representando as paredes
2. **Cobertura**: use loft para criar uma rampa de acesso
3. **Vãos**: abra pelo menos duas janelas e uma porta usando subtração
4. **Detalhe**: adicione colunas cilíndricas em pelo menos dois cantos usando união

Use variáveis para todas as dimensões e exiba com `show()` usando `visible_axes=None`, `z=0` e cores à sua escolha.

Consulte a lista completa de cores em [Plotly: Supported CSS Colors](https://plotly.com/python/css-colors/).

In [ ]:
# Escreva seu código aqui


---

## Resumo

Neste notebook você aprendeu:

- As primitivas do build123d: `Box`, `Cylinder`, `Sphere`
- O parâmetro `align` da caixa e o parâmetro `arc_size` do cilindro e da esfera
- Como criar **cúpulas** com `Sphere(arc_size1, arc_size2)`
- Como criar **cones e troncos** com a primitiva nativa `Cone(bottom_radius, top_radius, height)` — sem precisar recorrer ao OCCT diretamente
- Como criar **toros** com a primitiva nativa `Torus(major_radius, minor_radius)`
- Como criar **superfícies de revolução** com `BuildLine` + `BuildSketch` + `revolve()` — toros manuais, vasos e formas axialmente simétricas
- Como criar **rampas** via **loft** entre wires paralelos deslocados
- As três **operações booleanas**: união (`+` / `.fuse()`), subtração (`-` / `.cut()`) e interseção (`&` / `.intersect()`)
- Como **combinar** primitivas, loft, revolve e booleanas para construir modelos complexos

No próximo notebook veremos como trabalhar com **perfis 2D e extrusões** — a base da modelagem arquitetônica a partir de plantas baixas.

---
*build123d para Arquitetos e Engenheiros — Versão Google Colab*